In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
OUT_DIR = DATA_DIR / "oral_experiment"
META = ["participant_id", "cohort", "research_stage", "array_index", "collection_date"]


def clean_ids(s):
    return s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)


def to_date(values):
    return pd.to_datetime(values, errors="coerce", utc=True).dt.tz_convert(None).dt.normalize()


ids = clean_ids(pd.read_csv(DATA_DIR / "diet_experiment" / "diet_feature_matrix_baseline_full.csv",
                            usecols=["participant_id"])["participant_id"])
species = pd.read_csv(DATA_DIR / "csv_oral" / "species.csv", low_memory=False)
species["participant_id"] = clean_ids(species["participant_id"])
species = species[species["participant_id"].isin(ids)]

baseline = species["research_stage"].astype(str).str.strip().eq("00_00_visit")
if baseline.any():
    species = species[baseline]
else:
    cohort = pd.read_csv(DATA_DIR / "csv_diet" / "select_diet_oral.csv")
    oral_date = cohort.assign(participant_id=clean_ids(cohort["participant_id"])).set_index("participant_id")["oral_collection_date"]
    species = species[to_date(species["collection_date"]).eq(species["participant_id"].map(to_date(oral_date)))]
species = species.assign(_date=to_date(species["collection_date"])).sort_values(["participant_id", "_date"])
species = species.drop(columns="_date").drop_duplicates("participant_id")
OUT_DIR.mkdir(parents=True, exist_ok=True)
species.to_csv(OUT_DIR / "species_baseline_filtered.csv", index=False)

In [ ]:
abundance = species.drop(columns=[c for c in META if c in species]).apply(pd.to_numeric, errors="coerce").fillna(0).clip(lower=0)
p = abundance.div(abundance.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
p = p.where(p > 0)
diversity = pd.DataFrame({
    "participant_id": species["participant_id"].to_numpy(),
    "Richness": (abundance > 0).sum(axis=1).to_numpy(),
    "Shannon_diversity": -(p * np.log(p)).sum(axis=1).to_numpy(),
})
diversity.to_csv(OUT_DIR / "oral_diversity_baseline.csv", index=False)
print(diversity.describe())